### LangChain - ChatBot

1. Import libraries and load environment variables

In [ ]:
# !pip install python-dotenv streamlit pdfplumber langchain faiss-cpu 
# !pip install langchain-text-splitters
# !pip install --upgrade langchain
# !pip install langchain_community
# !pip install langchain-core
# !pip install langchain-huggingface sentence-transformers
# !pip install langchain-ollama
# !pip install faiss-cpu
# !pip install langchain-google-genai



import os                                                           # Import OS module to interact with environment variables and system functions
from dotenv import load_dotenv                                      # Load environment variables from a .env file
import streamlit as st                                              # Streamlit for building the web app UI
import pdfplumber 
from langchain_text_splitters import RecursiveCharacterTextSplitter # Split large text into smaller chunks
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores.faiss import FAISS            # FAISS vector store from langchain-community package
from langchain_core.documents import Document                       # Document class from langchain-core package
from langchain_classic.chains import RetrievalQA                    # RetrievalQA chain from langchain-classic package

load_dotenv()                                                       # Load environment variables from .env file




2. Initialize Embeddings and LLM

In [ ]:
# embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004")  # Initialize Google Generative AI embeddings model

from langchain_huggingface import HuggingFaceEmbeddings             # Initialize HuggingFace embeddings model  

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")            # Initialize HuggingFace embeddings model for generating vector representations of text

llm = ChatGoogleGenerativeAI(                                       # Initialize Google Generative AI chat model
    model="gemini-2.5-flash",   # free model                        # Specify the model to use for generating responses
    temperature=0                                                   # Set the temperature for response generation (controls randomness of output)
)

3. Set Up Streamlit UI

In [ ]:
st.title("PDF Question Answering with LangChain")                   # Set the title of the Streamlit app

uploaded_file = st.file_uploader("Upload a PDF", type=["pdf"])      # Create a file uploader for PDF files

4. Extract Text from Uploaded PDF

In [ ]:
full_text = ""                                                      # Initialize an empty string to hold the extracted text from the PDF
if uploaded_file is not None:                                       # If a PDF file has been uploaded, proceed to extract text from it
    with pdfplumber.open(uploaded_file) as pdf:                     # Open the uploaded PDF file using pdfplumber
        full_text = "".join(page.extract_text() or "" for page in pdf.pages)    # Extract text from each page of the PDF and concatenate it into a single string
if not full_text.strip():                                           # If no text was extracted from the PDF (i.e., the string is empty or contains only whitespace), display an error message and stop execution
        st.error("❌ No extractable text found. This PDF may be scanned or image-based.") # Display an error message
        st.stop()                                                   # Stop execution of the Streamlit app if no text was extracted from the PDF

5. Split Text into Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)  # Initialize a text splitter to divide the extracted text into smaller chunks of 1000 characters with an overlap of 100 characters between chunks
chunks = text_splitter.split_text(full_text)                        # Split the extracted text into smaller chunks using the text splitter

6. Create Document Objects for Vector Store

In [ ]:
docs = [Document(page_content=chunk) for chunk in chunks]           # Create a list of Document objects from the text chunks, where each chunk becomes a separate Document

7. Create FAISS Vector Store

In [ ]:
vectorstore = FAISS.from_documents(docs, embeddings)                # Create a FAISS vector store from the Document objects and their corresponding embeddings

8. Create Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})        # Create a retriever from the FAISS vector store to retrieve the top 3 most relevant documents based on a query

9. Create RetrievalQA Chain

In [ ]:
qa_chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)    # Create a RetrievalQA chain that combines the language model (llm) with the retriever to answer questions based on the retrieved documents

10. User Question Input and Display Answer

In [ ]:
question = st.text_input("Ask a question about the PDF")            # Create a text input field for the user to ask a question about the uploaded PDF

if question:                                                        # If the user has entered a question, proceed to run the QA chain to get an answer
    answer = qa_chain.run(question)                                 # Run the QA chain with the user's question to get an answer based on the retrieved documents
    st.write("### Answer")                                          # Display a heading for the answer section
    st.write(answer)                                                # Display the answer generated by the QA chain in the Streamlit app